# Reward Model Debug Notebook

Carica agente (`.zip`), reward model (`.pt`) e environment SUMO, raccoglie un rollout
tramite `TrajectoryGeneratorFromAgent`, e produce:
- **Scatter plot** true vs predicted return per segmenti di lunghezza fissa (con correlazioni Spearman / Pearson / Kendall)
- **Reward curve** per traiettoria (true vs predicted, interi episodi)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import pearsonr, spearmanr, kendalltau
from omegaconf import OmegaConf
from stable_baselines3 import PPO, SAC

import sumo_rl_ego as sre
from human_feedback_rl.common.reward_nets import RewardNet, RewardEnsemble
from human_feedback_rl.common.trajectory_generators import TrajectoryGeneratorFromAgent

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

ALGO_MAP = {"PPO": PPO, "SAC": SAC}

In [ ]:
# ── Modifica questi path con il run da analizzare ─────────────────────────────
RUN_DIR = Path("../outputs/checkpoint_0040")
N_STEPS = 2000   # step da raccogliere nel rollout

In [ ]:
cfg  = OmegaConf.load(RUN_DIR / "config.yaml")

print(f"Env:    {cfg.env.id}")
print(f"Agent:  {cfg.model.algo}")
print(f"Kwargs: {dict(cfg.env.kwargs)}")

venv = sre.make_vec_env(cfg.env.id, n_envs=1, base_seed=0, **cfg.env.kwargs)
print("Environment creato OK")

In [ ]:
# ── Caricamento reward model ──────────────────────────────────────────────────
# Ricostruisce l'architettura direttamente dal checkpoint, compatibile sia con
# il vecchio formato (obs+act, no bias sull'ultimo layer) che con il nuovo
# (obs+act+status+done, bias opzionale).

class _CompatibleMember(RewardNet):
    _STATUS_DIM = 7

    def __init__(self, obs_space, act_space, input_dim, hidden_dims, has_bias_last, obs_dim, act_dim):
        super().__init__(obs_space, act_space)
        self._uses_status = (input_dim == obs_dim + act_dim + self._STATUS_DIM + 1)
        layers, in_d = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(in_d, h), nn.Tanh()]
            in_d = h
        layers.append(nn.Linear(in_d, 1, bias=has_bias_last))
        self.net = nn.Sequential(*layers)

    def forward(self, state, action, next_status=None, done=None):
        if self._uses_status and next_status is not None and done is not None:
            x = torch.cat([state, action, next_status, done.unsqueeze(-1)], dim=1)
        else:
            x = torch.cat([state, action], dim=1)
        return self.net(x).squeeze(-1)


def _parse_member_arch(sd, member_idx=0):
    prefix = f"members.{member_idx}.net."
    wkeys  = sorted(
        [k for k in sd if k.startswith(prefix) and k.endswith(".weight")],
        key=lambda k: int(k[len(prefix):].split(".")[0]),
    )
    input_dim   = sd[wkeys[0]].shape[1]
    hidden_dims = [sd[k].shape[0] for k in wkeys[:-1]]
    last_idx    = wkeys[-1][len(prefix):].split(".")[0]
    has_bias    = f"{prefix}{last_idx}.bias" in sd
    return input_dim, hidden_dims, has_bias


def load_reward_model(path, obs_space, act_space):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    sd, n = ckpt["state_dict"], ckpt["n_members"]
    obs_dim, act_dim = ckpt["obs_dim"], ckpt["act_dim"]
    input_dim, hidden_dims, has_bias_last = _parse_member_arch(sd)
    uses_status = (input_dim == obs_dim + act_dim + 8)
    print(
        f"Checkpoint: {n} membri | input_dim={input_dim} "
        f"({'obs+act+status+done' if uses_status else 'obs+act'}) | "
        f"hidden={hidden_dims} | bias_last={has_bias_last}"
    )
    members = [
        _CompatibleMember(obs_space, act_space, input_dim, hidden_dims, has_bias_last, obs_dim, act_dim)
        for _ in range(n)
    ]
    ensemble = RewardEnsemble(obs_space, act_space, members)
    ensemble.load_state_dict(sd)
    ensemble.eval()
    return ensemble


reward_model = load_reward_model(
    RUN_DIR / "reward_model.pt",
    venv.observation_space,
    venv.action_space,
)
print("Reward model caricato OK")

In [ ]:
# ── Logger minimale (nessun wandb) ───────────────────────────────────────────
class _NullLogger:
    def log(self,  msg, *a, **kw): print(f"[rollout] {msg}")
    def warn(self, msg, *a, **kw): print(f"[rollout WARN] {msg}")
    def record(self, *a, **kw):    pass
    def dump(self,   *a, **kw):    pass
    def __getattr__(self, name):   return lambda *a, **kw: None


# ── Carica agente ─────────────────────────────────────────────────────────────
algo_cls = ALGO_MAP[cfg.model.algo]
agent    = algo_cls.load(RUN_DIR / "model.zip", device="cpu")
print(f"Agente caricato: {cfg.model.algo}")

# ── Crea trajectory generator ─────────────────────────────────────────────────
traj_gen = TrajectoryGeneratorFromAgent(
    agent=agent,
    reward_model=reward_model,
    venv=venv,
    rng=np.random.default_rng(42),
    logger=_NullLogger(),
)

# ── Rollout ───────────────────────────────────────────────────────────────────
print(f"Raccolta {N_STEPS} step...")
trajectories, log_metrics = traj_gen.sample(N_STEPS)
print("Done.")

In [ ]:
# ── Metriche rollout ──────────────────────────────────────────────────────────
print(f"Episodi:             {len(trajectories)}")
print(f"Step totali:         {sum(log_metrics.lengths)}")
print(f"Lunghezza media ep:  {log_metrics.mean_length:.1f}")
print(f"True reward medio:   {log_metrics.mean_true_reward:.3f}")
print(f"Model reward medio:  {log_metrics.mean_model_reward:.3f}")
print(f"Tempo campionamento: {log_metrics.time_sample:.1f} s")
print()
for i, (tr, mr, ln) in enumerate(
    zip(log_metrics.true_rewards, log_metrics.model_rewards, log_metrics.lengths)
):
    print(f"  Ep {i+1:2d}: len={ln:4d}  true_return={tr:8.2f}  model_return={mr:8.2f}")

In [ ]:
# ── Helper: estrazione segmenti ───────────────────────────────────────────────

def _extract_segments(trajs, segment_length):
    """Segmenti non sovrapposti di lunghezza fissa. None = episodio intero."""
    segments = []
    for traj in trajs:
        traj_list = list(traj)
        if segment_length is None:
            segments.append(traj_list)
        else:
            for idx in range(0, len(traj_list) - segment_length + 1, segment_length):
                seg = traj_list[idx : idx + segment_length]
                if len(seg) == segment_length:
                    segments.append(seg)
    return segments


def _score_segments(segments, rm):
    """Ritorna (true_returns, pred_returns) come np.ndarray."""
    true_returns, pred_returns = [], []
    for seg in segments:
        obs  = np.array([t.observation  for t in seg], dtype=np.float32)
        acts = np.array([t.action       for t in seg], dtype=np.float32)
        ns   = np.array([t.next_status  for t in seg], dtype=np.float32)
        dn   = np.array([float(t.done)  for t in seg], dtype=np.float32)
        pred_mean, _ = rm.predict_mean_std(obs, acts, ns, dn)
        true_returns.append(sum(t.true_reward for t in seg))
        pred_returns.append(float(pred_mean.sum()))
    return np.array(true_returns), np.array(pred_returns)


# ── Scatter plot ──────────────────────────────────────────────────────────────

def plot_scatter(trajs, rm, segment_length, z_standardize):
    """
    Scatter plot: true return vs predicted return per segmenti.

    segment_length : int oppure None (None = episodio intero)
    z_standardize  : bool — se True applica z-score a ciascun asse
    """
    segments = _extract_segments(trajs, segment_length)
    if not segments:
        print(f"Nessun segmento di lunghezza {segment_length} trovato.")
        return

    true_ret, pred_ret = _score_segments(segments, rm)

    if z_standardize:
        true_ret = (true_ret - true_ret.mean()) / (true_ret.std() + 1e-8)
        pred_ret = (pred_ret - pred_ret.mean()) / (pred_ret.std() + 1e-8)

    pearson_r,    _ = pearsonr(true_ret, pred_ret)
    spearman_rho, _ = spearmanr(true_ret, pred_ret)
    kendall_tau,  _ = kendalltau(true_ret, pred_ret)

    label = f"seg={segment_length}" if segment_length is not None else "episodio intero"
    z_tag = " (z-standardized)" if z_standardize else ""

    combined = np.concatenate([true_ret, pred_ret])
    span = combined.max() - combined.min() + 1e-8
    lo = combined.min() - 0.05 * span
    hi = combined.max() + 0.05 * span

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(true_ret, pred_ret, alpha=0.55, s=25, edgecolors="none")
    ax.plot([lo, hi], [lo, hi], "r--", lw=1.5, label="y=x")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel(f"True return{z_tag}")
    ax.set_ylabel(f"Predicted return{z_tag}")
    ax.set_title(
        f"Scatter — {label}  (n={len(segments)})\n"
        f"Pearson r={pearson_r:.3f}   Spearman ρ={spearman_rho:.3f}   Kendall τ={kendall_tau:.3f}"
    )
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(
        f"[{label}] Pearson={pearson_r:.3f}  "
        f"Spearman={spearman_rho:.3f}  Kendall={kendall_tau:.3f}  n={len(segments)}"
    )


# ── Reward curve per traiettoria ──────────────────────────────────────────────

def plot_reward_curves(trajs, rm, z_standardize):
    """
    True vs predicted reward step-by-step per ogni episodio.

    z_standardize : bool — se True, z-normalizza usando statistiche globali
                    (stesso mean/std calcolato su tutti gli step di tutti gli episodi)
    """
    pred_per_traj = []
    for traj in trajs:
        obs  = np.array([t.observation  for t in traj], dtype=np.float32)
        acts = np.array([t.action       for t in traj], dtype=np.float32)
        ns   = np.array([t.next_status  for t in traj], dtype=np.float32)
        dn   = np.array([float(t.done)  for t in traj], dtype=np.float32)
        pm, ps = rm.predict_mean_std(obs, acts, ns, dn)
        pred_per_traj.append((pm, ps))

    all_true = np.concatenate([[t.true_reward for t in traj] for traj in trajs])
    all_pred = np.concatenate([pm for pm, _ in pred_per_traj])

    tm, ts, pm_g, ps_g = 0.0, 1.0, 0.0, 1.0   # defaults (unused if not z_standardize)
    if z_standardize:
        tm,   ts   = all_true.mean(), all_true.std() + 1e-8
        pm_g, ps_g = all_pred.mean(), all_pred.std() + 1e-8

    n = len(trajs)
    fig, axes = plt.subplots(n, 1, figsize=(12, 3 * n), sharex=False)
    if n == 1:
        axes = [axes]

    for ep_idx, (traj, ax, (pred_mean, pred_std)) in enumerate(
        zip(trajs, axes, pred_per_traj)
    ):
        true_r = np.array([t.true_reward for t in traj])
        pred_r = pred_mean.copy()
        pred_s = pred_std.copy()

        if z_standardize:
            true_r = (true_r - tm)   / ts
            pred_r = (pred_r - pm_g) / ps_g
            pred_s = pred_s / (ps_g + 1e-8)

        t_ax = np.arange(len(true_r))
        ax.plot(t_ax, true_r, label="True",      color="royalblue", lw=1.5)
        ax.plot(t_ax, pred_r, label="Predicted",  color="tomato",    lw=1.5, ls="--")
        ax.fill_between(
            t_ax, pred_r - pred_s, pred_r + pred_s,
            alpha=0.2, color="tomato", label="±1σ ensemble"
        )

        true_ret_val = sum(transition.true_reward for transition in traj)
        pred_ret_val = float(pred_mean.sum())
        z_tag = " (z)" if z_standardize else ""
        ax.set_title(f"Ep {ep_idx+1}  |  true_return={true_ret_val:.1f}  pred_return={pred_ret_val:.1f}")
        ax.set_xlabel("Step")
        ax.set_ylabel(f"Reward{z_tag}")
        ax.legend(loc="upper right", fontsize=8)

    z_label = "z-standardized" if z_standardize else "raw"
    plt.suptitle(f"Reward curves — {z_label}", y=1.002, fontsize=13)
    plt.tight_layout()
    plt.show()


print("Funzioni di plot definite OK")

In [ ]:
# ── Scatter plot per diverse lunghezze di segmento ────────────────────────────
for seg_len in [1, 5, 20, None]:
    plot_scatter(trajectories, reward_model, segment_length=seg_len, z_standardize=False)

In [ ]:
plot_reward_curves(trajectories, reward_model, z_standardize=False)

In [ ]:
plot_reward_curves(trajectories, reward_model, z_standardize=True)